# EXP-005C — Level-to-Level Transfer from EXP-005A

Planner-focused follow-up to EXP-005A. EXP-005B broadened the action scan and regressed, so this notebook returns to the EXP-005A backbone and adds level-to-level solution transfer.

Core idea: if BFS solved an earlier level, try replaying that solution directly on the next level. If direct replay fails, try an object-relative click offset transfer.

Expected artifacts: `/kaggle/working/exp005c_bfs_events.jsonl` and `/kaggle/working/exp005c_run_summary.json`.

Current reference: EXP-005A Version 9 public score `0.17`.


In [ ]:
!pip install -q --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
import copy, glob, hashlib, importlib.util, json, os, random, time, traceback
from collections import Counter, deque
from pathlib import Path
import numpy as np
from agents.agent import Agent
from arcengine import ActionInput, GameAction, GameState
BFS_EVENT_PATH=Path('/kaggle/working/exp005c_bfs_events.jsonl')
RUN_SUMMARY_PATH=Path('/kaggle/working/exp005c_run_summary.json')
def logj(x):
    try:
        BFS_EVENT_PATH.parent.mkdir(parents=True,exist_ok=True); BFS_EVENT_PATH.open('a').write(json.dumps(x,default=str)+'\n')
    except Exception: pass
def savej(x):
    try: RUN_SUMMARY_PATH.write_text(json.dumps(x,indent=2,default=str))
    except Exception: pass
def aid(a):
    try: return int(a.value)
    except Exception: return int(a)
def lf(obj):
    fr=getattr(obj,'frame',None)
    if fr is None: return None
    ar=np.asarray(fr)
    return ar[-1].astype(np.int64) if ar.ndim==3 else ar.astype(np.int64) if ar.ndim==2 else None
def ai(i,data=None):
    g=GameAction.from_id(int(i)); return ActionInput(id=g,data=dict(data)) if data else ActionInput(id=g)
def act(i,data=None):
    a=GameAction.from_id(int(i))
    if data: a.set_data(dict(data))
    return a
def find_src(game_id,arc_env=None):
    gid=str(game_id).split('-')[0]; guess=(gid[0].upper()+gid[1:]) if gid and gid[0].isalpha() else gid.capitalize()
    def cls(path):
        try:
            import re; txt=Path(path).read_text(errors='ignore')[:5000]; m=re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame',txt); return m.group(1) if m else guess
        except Exception: return guess
    try:
        d=getattr(getattr(arc_env,'environment_info',None),'local_dir',None)
        if d:
            r=Path(d); c=[r/f'{gid}.py',r/f'{gid.lower()}.py',r/f'{guess.lower()}.py']+list(r.rglob(f'{gid}.py'))[:25]+list(r.rglob(f'{gid.lower()}.py'))[:25]
            for p in c:
                if p.exists(): return str(p),cls(p)
    except Exception: pass
    for pat in [f'/tmp/**/{gid}.py',f'/kaggle/working/**/{gid}.py',f'/kaggle/input/**/{gid}.py',f'**/game_sources/**/{gid}.py']:
        try:
            m=glob.glob(pat,recursive=True)
            if m: return m[0],cls(m[0])
        except Exception: pass
    return None,guess
def simple_objects(frame,bg):
    objs=[]
    for c in range(16):
        if c==bg: continue
        mask=(frame==c); n=int(np.sum(mask))
        if n<2 or n>3000: continue
        ys,xs=np.where(mask); objs.append({'color':c,'n':n,'cx':float(np.mean(xs)),'cy':float(np.mean(ys))})
    return sorted(objs,key=lambda o:(o['color'],-o['n']))
class Solver:
    def __init__(s,path,cls_name,game_id='unknown'):
        s.path=path; s.cls_name=cls_name; s.game_id=game_id; s.game_cls=None; s.last_log={}; s.solutions={}
        s.scan_timeout=float(os.getenv('EXP005C_SCAN_TIMEOUT','4')); s.bfs_timeout=float(os.getenv('EXP005C_BFS_TIMEOUT','25')); s.max_states=int(os.getenv('EXP005C_MAX_STATES','50000')); s.max_depth=int(os.getenv('EXP005C_MAX_DEPTH','30'))
    def load(s):
        try:
            spec=importlib.util.spec_from_file_location('exp005c_game',s.path); mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); s.game_cls=getattr(mod,s.cls_name); return True
        except Exception as e:
            s.last_log={'event':'load','variant':'EXP-005C','status':'error','game_id':s.game_id,'path':s.path,'class':s.cls_name,'error':repr(e)}; logj(s.last_log); return False
    def reset(s,lvl):
        g=s.game_cls(); g.set_level(int(lvl)); g.perform_action(ai(GameAction.RESET.value),raw=True); r=g.perform_action(ai(GameAction.RESET.value),raw=True); f=lf(r)
        if f is None: f=np.asarray(g.get_pixels(0,0,64,64),dtype=np.int64)
        return g,f
    def fh(s,f): return hashlib.md5(np.asarray(f,dtype=np.uint8).tobytes()).hexdigest()
    def scalars(s,g):
        ex={'_action_count','_full_reset','_action_complete','_last_action','_last_frame','_rng','rng','np_random'}; out={}
        for k,v in getattr(g,'__dict__',{}).items():
            if k in ex or k.startswith('__'): continue
            if isinstance(v,(int,float,bool,str)) and (not isinstance(v,str) or len(v)<=80): out[k]=v
        return out
    def sh(s,g,f,hid):
        h=s.fh(f); vals=[]
        for k in hid:
            try: vals.append(f'{k}={getattr(g,k,None)}')
            except Exception: pass
        return h+'|'+'|'.join(vals) if vals else h
    def av(s,g):
        out=[]
        for a in getattr(g,'_available_actions',[]):
            try:
                x=aid(a)
                if x not in out: out.append(x)
            except Exception: pass
        return out
    def scan(s,g0,f0):
        acts=[]; bg=int(np.bincount(f0.flatten(),minlength=16).argmax()); ids=s.av(g0); stats={'available':ids,'dir':0,'click':0,'dedup':0,'total':0}
        for a in [x for x in ids if 1<=x<=5]:
            try:
                g=copy.deepcopy(g0); r=g.perform_action(ai(a),raw=True); f=lf(r)
                if f is not None and np.sum(f0!=f)>0: acts.append((a,None)); stats['dir']+=1
            except Exception: pass
        if 6 in ids:
            t=time.time(); seen=set()
            for y in range(0,64,2):
                if time.time()-t>s.scan_timeout: break
                for x in range(0,64,2):
                    if f0[y,x]==bg: continue
                    data={'x':int(x),'y':int(y),'game_id':'exp005c_bfs'}
                    try:
                        g=copy.deepcopy(g0); r=g.perform_action(ai(6,data),raw=True); f=lf(r)
                        if f is not None and np.sum(f0!=f)>0:
                            eh=s.fh(f)
                            if eh not in seen: seen.add(eh); acts.append((6,data)); stats['click']+=1
                            else: stats['dedup']+=1
                    except Exception: pass
        stats['total']=len(acts); return acts,stats
    def probe(s,g0,acts):
        base=s.scalars(g0); ch=set()
        for a,d in acts[:12]:
            try:
                g=copy.deepcopy(g0); g.perform_action(ai(a,d),raw=True); aft=s.scalars(g)
                for k,v in base.items():
                    if k in aft and aft[k]!=v: ch.add(k)
            except Exception: pass
        return [k for k in sorted(ch) if not any(n in k.lower() for n in ('time','timestamp','elapsed','duration'))]
    def run_hist(s,lvl,hist):
        g,f0=s.reset(lvl)
        for idx,(a,d) in enumerate(hist):
            try:
                r=g.perform_action(ai(a,d),raw=True)
                cur=getattr(g,'_current_level_index',lvl); done=getattr(r,'levels_completed',lvl)
                if done>lvl or cur>lvl: return hist[:idx+1], 'solved'
            except Exception:
                return None,'error'
        return None,'not_solved'
    def transfer_solution(s,lvl,f_curr,prev_hist):
        if not prev_hist or lvl<=0: return None, {'attempted':False}
        # 1) direct replay
        sol,status=s.run_hist(lvl,prev_hist)
        info={'attempted':True,'direct_status':status,'offset_status':None,'dx':None,'dy':None}
        if sol: return sol,info
        # 2) object-relative click offset
        try:
            _,f_prev=s.reset(lvl-1); bg_prev=int(np.bincount(f_prev.flatten(),minlength=16).argmax()); bg_curr=int(np.bincount(f_curr.flatten(),minlength=16).argmax())
            op=simple_objects(f_prev,bg_prev); oc=simple_objects(f_curr,bg_curr); pairs=[]
            for p in op:
                best=None; bd=10**9
                for c in oc:
                    if c['color']==p['color'] and abs(c['n']-p['n'])<=max(2,0.5*max(c['n'],p['n'])):
                        d=abs(c['cx']-p['cx'])+abs(c['cy']-p['cy'])
                        if d<bd: bd=d; best=c
                if best: pairs.append((p,best))
            if not pairs: info['offset_status']='no_matches'; return None,info
            dx=float(np.mean([b['cx']-a['cx'] for a,b in pairs])); dy=float(np.mean([b['cy']-a['cy'] for a,b in pairs])); info['dx']=round(dx,2); info['dy']=round(dy,2)
            shifted=[]
            for a,d in prev_hist:
                if d and 'x' in d and 'y' in d:
                    nd=dict(d); nd['x']=int(max(0,min(63,round(nd.get('x',32)+dx)))); nd['y']=int(max(0,min(63,round(nd.get('y',32)+dy)))); nd['game_id']='exp005c_transfer'
                    shifted.append((a,nd))
                else: shifted.append((a,d))
            sol,status=s.run_hist(lvl,shifted); info['offset_status']=status
            if sol: return sol,info
            return None,info
        except Exception as e:
            info['offset_status']='error'; info['error']=repr(e); return None,info
    def solve(s,lvl):
        t=time.time(); log={'event':'solve','variant':'EXP-005C','game_id':s.game_id,'level':int(lvl),'path':s.path,'class':s.cls_name,'status':'started','scan':{},'hidden':[],'transfer':{},'explored':0,'unique':0,'solution_len':None}
        try:
            base,f0=s.reset(lvl)
            prev=s.solutions.get(lvl-1)
            if prev:
                sol,info=s.transfer_solution(lvl,f0,prev); log['transfer']=info
                if sol:
                    s.solutions[lvl]=sol; log.update({'status':'transfer_solved','solution_len':len(sol),'elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return sol
            acts,st=s.scan(base,f0); hid=s.probe(base,acts); log.update({'scan':st,'hidden':hid,'hidden_hash':bool(hid)})
            if not acts: log['status']='no_effective_actions'; log['elapsed']=round(time.time()-t,3); s.last_log=log; logj(log); return None
            q=deque([(copy.deepcopy(base),[],0)]); seen={s.sh(base,f0,hid)}
            while q and log['explored']<s.max_states and time.time()-t<s.bfs_timeout:
                g,hist,depth=q.popleft()
                for a,d in acts:
                    if log['explored']>=s.max_states or time.time()-t>=s.bfs_timeout: break
                    try:
                        g2=copy.deepcopy(g); r=g2.perform_action(ai(a,d),raw=True); log['explored']+=1; f=lf(r)
                        if f is None: continue
                        hh=s.sh(g2,f,hid)
                        if hh in seen: continue
                        seen.add(hh); nh=hist+[(a,d)]; cur=getattr(g2,'_current_level_index',lvl); done=getattr(r,'levels_completed',lvl)
                        if done>lvl or cur>lvl:
                            s.solutions[lvl]=nh; log.update({'unique':len(seen),'solution_len':len(nh),'status':'bfs_solved','elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return nh
                        if depth+1<s.max_depth: q.append((g2,nh,depth+1))
                    except Exception: pass
            log.update({'unique':len(seen),'status':'timeout_or_exhausted','elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return None
        except Exception as e:
            log.update({'status':'error','error':repr(e),'elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return None
class MyAgent(Agent):
    MAX_ACTIONS=float('inf'); _MAX_FRAMES=10
    def __init__(s,*a,**kw):
        super().__init__(*a,**kw); seed=int(time.time()*1e6)^(hash(getattr(s,'game_id','unknown'))&0xffffffff); s.rng=random.Random(seed); np.random.seed(seed%(2**32-1))
        s.current_level=None; s.bfs=None; s.bfs_ready=False; s.src=None; s.cls=None; s.solution=None; s.step=0; s.tried=set(); s.policy=Counter(); s.actions=Counter(); s.logs=[]; s.start=time.time()
    def summary(s): savej({'variant':'EXP-005C','game_id':getattr(s,'game_id',None),'source':s.src,'class':s.cls,'levels':sorted(list(s.tried)),'policy':dict(s.policy),'actions':dict(s.actions),'logs':s.logs[-40:]})
    def append_frame(s,f):
        s.frames.append(f); s.frames=s.frames[-s._MAX_FRAMES:]
        if getattr(f,'guid',None): s.guid=f.guid
        if hasattr(s,'recorder') and not s.is_playback:
            try: s.recorder.record(json.loads(f.model_dump_json()))
            except Exception: pass
    def level(s,lfx): return int(getattr(lfx,'levels_completed',0) or 0)
    def raw(s,lfx):
        f=lf(lfx); return f if f is not None else np.zeros((64,64),dtype=np.int64)
    def avail(s,lfx):
        out=[]
        for a in getattr(lfx,'available_actions',None) or []:
            try:
                x=aid(a)
                if x not in out: out.append(x)
            except Exception: pass
        return out
    def init_bfs(s):
        if s.bfs_ready: return
        s.src,s.cls=find_src(s.game_id,getattr(s,'arc_env',None))
        if s.src:
            s.bfs=Solver(s.src,s.cls,game_id=getattr(s,'game_id','unknown'))
            if not s.bfs.load(): s.bfs=None
        if s.bfs is None:
            p={'event':'bfs_init','variant':'EXP-005C','status':'source_not_found_or_load_failed','game_id':getattr(s,'game_id',None),'source':s.src,'class':s.cls}; s.logs.append(p); logj(p)
        s.bfs_ready=True; s.summary()
    def try_bfs(s,lvl):
        if lvl in s.tried: return
        s.tried.add(lvl); s.init_bfs()
        if s.bfs is None: return
        sol=s.bfs.solve(lvl); s.logs.append(dict(s.bfs.last_log))
        if sol: s.solution=sol; s.step=0
        s.summary()
    def click(s,raw):
        bg=int(np.bincount(raw.flatten(),minlength=16).argmax()); ys,xs=np.where(raw!=bg)
        if len(xs)==0: return None
        i=s.rng.randrange(len(xs)); return {'x':int(xs[i]),'y':int(ys[i]),'game_id':'exp005c_fallback'}
    def fallback(s,raw,lfx):
        av=[x for x in s.avail(lfx) if x in [1,2,3,4,5,6,7]]
        if 6 in av and s.rng.random()<0.65:
            c=s.click(raw)
            if c:
                a=GameAction.ACTION6; a.set_data(c); a.reasoning='fallback:visible_click'; return a,6
        mv=[x for x in av if 1<=x<=5]
        if mv:
            x=s.rng.choice(mv); a=act(x); a.reasoning='fallback:random_move'; return a,x
        if 7 in av:
            a=act(7); a.reasoning='fallback:undo_last_resort'; return a,7
        a=GameAction.RESET; a.reasoning='fallback:reset_last_resort'; return a,0
    def is_done(s,frames,lfx):
        try:
            d=lfx.state is GameState.WIN or time.time()-s.start>=8*3600-300
            if d: s.summary()
            return d
        except Exception: s.summary(); return True
    def choose_action(s,frames,lfx):
        try:
            lvl=s.level(lfx); raw=s.raw(lfx); st=getattr(lfx,'state',None)
            if st in [GameState.NOT_PLAYED,GameState.GAME_OVER]: s.policy['reset']+=1; a=GameAction.RESET; a.reasoning='reset'; return a
            if lvl!=s.current_level: s.current_level=lvl; s.solution=None; s.step=0; s.try_bfs(lvl)
            if s.solution and s.step<len(s.solution):
                i,d=s.solution[s.step]; s.step+=1; a=act(i,d); a.reasoning=f'bfs_or_transfer_replay:{s.step}/{len(s.solution)}'; s.policy['bfs_or_transfer_replay']+=1; s.actions[int(i)]+=1; s.summary(); return a
            a,i=s.fallback(raw,lfx); s.policy['fallback']+=1; s.actions[int(i)]+=1; return a
        except Exception as e:
            traceback.print_exc(); s.policy['error_fallback']+=1; s.summary(); a=random.choice([GameAction.ACTION1,GameAction.ACTION2,GameAction.ACTION3,GameAction.ACTION4,GameAction.ACTION5]); a.reasoning=f'error_fallback:{e}'; return a


In [ ]:
import os, shutil, subprocess
from pathlib import Path
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    BASE=Path('/kaggle/working/ARC-AGI-3-Agents'); SRC=Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents')
    AGENT_SRC=Path('/kaggle/working/my_agent.py'); AGENT_DST=BASE/'agents/templates/my_agent.py'
    subprocess.run(['curl','--fail','--retry','60','--retry-all-errors','--retry-delay','5','--retry-max-time','300','http://gateway:8001/api/games'], check=False)
    if BASE.exists(): shutil.rmtree(BASE)
    shutil.copytree(SRC,BASE); shutil.copy(AGENT_SRC,AGENT_DST)
    (BASE/'agents/__init__.py').write_text('from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}\n')
    (BASE/'.env').write_text('\n'.join(['SCHEME=http','HOST=gateway','PORT=8001','ARC_API_KEY=test-key-123','ARC_BASE_URL=http://gateway:8001/','OPERATION_MODE=online','RECORDINGS_DIR=/kaggle/working/server_recording'])+'\n')
    subprocess.run(['python','main.py','--agent','myagent'],cwd=str(BASE),env={**os.environ,'MPLBACKEND':'agg','PYTHONUNBUFFERED':'1'},check=True)
else:
    print('Local/non-rerun mode: writing dummy submission fallback.')


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame([{'row_id':'debug_0','game_id':'1','end_of_game':True,'score':0}]).to_parquet('/kaggle/working/submission.parquet',index=False)
    print('Wrote /kaggle/working/submission.parquet dummy fallback.')


In [ ]:
from pathlib import Path
import json, os
BFS_EVENT_PATH=Path('/kaggle/working/exp005c_bfs_events.jsonl')
RUN_SUMMARY_PATH=Path('/kaggle/working/exp005c_run_summary.json')
if not BFS_EVENT_PATH.exists():
    BFS_EVENT_PATH.write_text(json.dumps({'variant':'EXP-005C','event':'artifact_placeholder','mode':'non_rerun_notebook_mode','reason':'KAGGLE_IS_COMPETITION_RERUN was not set, so main.py --agent myagent did not run.','expected_real_artifact_when_scored':str(BFS_EVENT_PATH)})+'\n')
if not RUN_SUMMARY_PATH.exists():
    RUN_SUMMARY_PATH.write_text(json.dumps({'variant':'EXP-005C','mode':'non_rerun_notebook_mode','status':'placeholder_only','reason':'Competition gateway did not run in visible notebook mode.','current_best_reference':{'experiment':'EXP-005A','public_score':0.17,'version':9},'expected_real_fields_when_scored':['source','class','levels','policy','actions','logs','transfer.direct_status','transfer.offset_status','transfer.dx','transfer.dy','hidden','hidden_hash','status','solution_len']},indent=2))
print('EXP-005C artifact check:')
print(' -',BFS_EVENT_PATH,'exists:',BFS_EVENT_PATH.exists(),'size:',BFS_EVENT_PATH.stat().st_size)
print(' -',RUN_SUMMARY_PATH,'exists:',RUN_SUMMARY_PATH.exists(),'size:',RUN_SUMMARY_PATH.stat().st_size)


## Tracker row draft

```text
| EXP-005C | 2026-05-21 | `notebooks/01_exploration/exp005c_level_transfer_from_exp005a.ipynb` | Level-to-level transfer + hidden hash | EXP-005A backbone plus direct replay and object-offset transfer from previous-level solutions | Pending | Pending | Created / awaiting validation | Goal: improve planner reuse without broadening action scan. |
```
